<a href="https://colab.research.google.com/github/PrantoMondol11/Machine-learning-/blob/main/VGG16_retraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 # IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
zalando_research_fashionmnist_path = kagglehub.dataset_download('zalando-research/fashionmnist')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd

In [ ]:
csv_file="/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_train.csv"


In [ ]:
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
from PIL import Image
import numpy as np
import torch.optim as optim

In [ ]:
class CustomClass(Dataset):
    def __init__(self,csv_file,transform):
        self.csv_file=csv_file
        self.transform=transform
        df=pd.read_csv(self.csv_file)
        self.X_train=df.iloc[:,1:].values
        self.y_train=df.iloc[:,0]
    def __len__(self):
        return len(self.X_train)

    def __getitem__(self,idx):
        image=self.X_train[idx].reshape(28,28)
        image=image.astype(np.uint8)
        image=np.stack([image]*3,axis=-1)
        image=Image.fromarray(image)
        image=self.transform(image)

        return image,torch.tensor(self.y_train[idx],dtype=torch.long)

In [ ]:
import torch.nn as nn
import torch
from torchvision.transforms import transforms
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
     transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
   ]
)

In [ ]:
import torchvision.models as models
vgg16= models.vgg16(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
lr=0.001
epochs=10
import torch.nn as nn

In [ ]:
for param in vgg16.features.parameters():
    param.requires_grad=False

In [ ]:
vgg16.classifier=nn.Sequential(
    nn.Linear(25088,1024),
    nn.ReLU(),
    nn.Dropout(.5),
    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Dropout(.5),

    nn.Linear(512,10)
)

In [ ]:
import torch
import torch.optim
device =torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = vgg16.to(device)

In [ ]:
loss1=nn.CrossEntropyLoss()
optimizer=optim.Adam(vgg16.classifier.parameters(),lr=lr)

In [ ]:
import pandas as pd
csv_file2="/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_test.csv"
dataset=CustomClass(csv_file,transform)
train_loader=DataLoader(dataset,batch_size=32,shuffle=True)
dataset2=CustomClass(csv_file2,transform)
test_loader=DataLoader(dataset2,batch_size=32,shuffle=True)

In [ ]:
for epoch in range(epochs):
    for batch_features,batch_labels in train_loader:
        batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)
        output=vgg16(batch_features)
        loss=loss1(output,batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()



In [ ]:
with torch.no_grad():
    total=0
    correct=0
    for batch_features,batch_labels in test_loader:
        batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)
        output=vgg16(batch_features)
        _,predicted=torch.max(output,1)
        total+=batch_features.shape[0]
        correct+=(predicted==batch_labels).sum().item()
    accuracy=correct/total

print(accuracy)


0.9033
